In [1]:
import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import json
import pickle

In [2]:
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras.optimizers import SGD
import random

In [3]:
words=[]
labels = []
documents = []
ignore_words = ['?', '!']


In [5]:
data_file = open('/content/sample_data/intents.json').read()
intents = json.loads(data_file)

In [6]:
# FIX 1: Ensure all NLTK data is downloaded properly
import nltk
import ssl

# Handle SSL certificate issues (common in some environments)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download required NLTK data
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [10]:

# Your code (now fixed)
import json

# Load your intents JSON
with open('/content/sample_data/intents.json', 'r') as f:
    intents = json.load(f)

words = []
documents = []

for intent in intents['intents']:
    for pattern in intent['patterns']:
        # Tokenize each word
        w = nltk.word_tokenize(pattern)
        words.extend(w)

        # Add documents in the corpus
        documents.append((w, intent['tag']))

print(f"✓ Loaded {len(documents)} patterns")
print(f"✓ Found {len(set(words))} unique words")


✓ Loaded 113 patterns
✓ Found 145 unique words


In [11]:
import nltk
nltk.download('wordnet')
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
# sort labels
labels = sorted(list(set(labels)))
# documents = combination between patterns and intents
print (len(documents), "documents")
# labels = intents
print (len(labels), "labels", labels)
# words = all words, vocabulary
print (len(words), "unique lemmatized words", words)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


113 documents
31 labels ['Identity', 'activity', 'age', 'appreciate', 'contact', 'covid19', 'cricket', 'datetime', 'exclaim', 'goodbye', 'google', 'greeting', 'greetreply', 'haha', 'inspire', 'insult', 'jokes', 'karan', 'news', 'nicetty', 'no', 'noanswer', 'options', 'programmer', 'riddle', 'song', 'suggest', 'thanks', 'timer', 'weather', 'whatsup']
130 unique lemmatized words ["'m", "'s", ',', '10', '19', 'a', 'age', 'am', 'anyone', 'are', 'ask', 'awesome', 'bad', 'bbye', 'be', 'best', 'bye', 'can', 'contact', 'could', 'covid', 'creator', 'cricket', 'current', 'date', 'day', 'designed', 'developer', 'do', 'doing', 'dumb', 'fine', 'for', 'funny', 'get', 'good', 'goodbye', 'google', 'great', 'haha', 'he', 'hello', 'help', 'helpful', 'helping', 'hey', 'hi', 'hola', 'hot', 'how', 'i', 'idiot', 'india', 'inspiration', 'inspires', 'internet', 'is', 'it', 'joke', 'karan', 'know', 'later', 'latest', 'laugh', 'lmao', 'lol', 'lost', 'made', 'make', 'malik', 'match', 'me', 'motivates', 'namaste'

In [12]:
pickle.dump(words,open('/content/sample_data/words.pkl','wb'))
pickle.dump(labels,open('/content/sample_data/labels.pkl','wb'))

In [13]:
labels = [intent['tag'] for intent in intents['intents']]
print(labels)  # Check if 'google' is here

['google', 'greeting', 'goodbye', 'thanks', 'noanswer', 'options', 'jokes', 'Identity', 'datetime', 'whatsup', 'haha', 'programmer', 'insult', 'activity', 'exclaim', 'weather', 'karan', 'contact', 'appreciate', 'nicetty', 'no', 'news', 'inspire', 'cricket', 'song', 'greetreply', 'timer', 'covid19', 'suggest', 'riddle', 'age']


In [14]:
# Check what's actually in documents vs labels
for doc in documents:
    tag = doc[1]
    if tag not in labels:
        print(f"Found in documents but NOT in labels: '{tag}'")
        print(f"  Raw representation: {repr(tag)}")  # Shows hidden chars
        print(f"  Length: {len(tag)}")

In [15]:
# create our training data
training = []
# create an empty array for our output
output_empty = [0] * len(labels)
# training set, bag of words for each sentence
for doc in documents:
    # initialize our bag of words
    bag = []
    # list of tokenized words for the pattern
    pattern_words = doc[0]
    # lemmatize each word - create base word, in attempt to represent related words
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    # create our bag of words array with 1, if word match found in current pattern
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    # output is a '0' for each tag and '1' for current tag (for each pattern)
    output_row = list(output_empty)
    output_row[labels.index(doc[1])] = 1  # ← This line was missing


    training.append([bag, output_row])
# shuffle our features and turn into np.array
random.shuffle(training)

train_x = [item[0] for item in training]
train_y = [item[1] for item in training]

train_x = np.array(train_x)
train_y = np.array(train_y)


In [16]:
# Create model - 3 layers. First layer 128 neurons, second layer 64 neurons and 3rd output layer contains number of neurons
# equal to number of intents to predict output intent with softmax
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
from tensorflow.keras.optimizers import SGD

sgd = SGD(
    learning_rate=0.01,
    momentum=0.9,
    nesterov=True
)

model.compile(
    loss='categorical_crossentropy',
    optimizer=sgd,
    metrics=['accuracy']
)



In [18]:



model.save('chatbot_model.h5')

print("model created")

model created


In [19]:
import nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import pickle
import numpy as np

In [20]:

import json
import random

In [21]:
def clean_up_sentence(sentence):
    # tokenize the pattern - split words into array
    sentence_words = nltk.word_tokenize(sentence)
    # stem each word - create short form for word
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

In [25]:
def bow(sentence, words, show_details=True):
    # tokenize the pattern
    sentence_words = clean_up_sentence(sentence)
    # bag of words - matrix of N words, vocabulary matrix
    bag = [0]*len(words)
    for s in sentence_words:
        for i,w in enumerate(words):
            if w == s:
                # assign 1 if current word is in the vocabulary position
                bag[i] = 1
                if show_details:
                    print ("found in bag: %s" % w)
    return(np.array(bag))


In [26]:
def predict_class(sentence, model):
    p = bow(sentence, words, show_details=False)

    res = model.predict(np.array([p]), verbose=0)[0]

    print("Raw prediction:", res)
    print("Max probability:", max(res))

    ERROR_THRESHOLD = 0.25

    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]

    results.sort(key=lambda x: x[1], reverse=True)

    return_list = []
    for r in results:
        return_list.append({
            "intent": labels[r[0]],
            "probability": str(r[1])
        })

    return return_list

In [27]:
def getResponse(ints, intents_json):
    if not ints:
        return "Sorry, I don't understand."

    tag = ints[0]['intent']

    for intent in intents_json['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

    return "Sorry, I don't understand."

In [28]:
def chatbot_response(msg):
    ints = predict_class(msg, model)
    res = getResponse(ints, intents)
    return res

In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        16,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 31)             │         2,015 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,039 (105.62 KB)

 Trainable params: 27,039 (105.62 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
print(model.metrics_names)
loss, acc = model.evaluate(train_x, train_y, verbose=0)
print("Training accuracy:", acc)
print("Training loss:", loss)

['loss', 'compile_metrics']
Training accuracy: 0.017699114978313446
Training loss: 3.4531474113464355


In [31]:
hist = model.fit(
    train_x,
    train_y,
    epochs=200,
    batch_size=5,
    verbose=1
)

Epoch 1/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.0442 - loss: 3.4722   
Epoch 2/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0619 - loss: 3.4194 
Epoch 3/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0708 - loss: 3.3595 
Epoch 4/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1150 - loss: 3.3048     
Epoch 5/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1150 - loss: 3.2691     
Epoch 6/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1416 - loss: 3.2121     
Epoch 7/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1327 - loss: 3.1651 
Epoch 8/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2124 - loss: 3.0514     
Epoch 9/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2124 - loss: 2.9895     
Epoch 10/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2035 - loss: 2.9095 
Epoch 11/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3009 - loss: 2.7782 
Epoch 12/200
23/23 ━━━━━━━━━━━

In [32]:
loss, acc = model.evaluate(train_x, train_y, verbose=0)
print("Accuracy:", acc)

Accuracy: 0.991150438785553


In [33]:
kill_siri = False

while kill_siri != True:
    msg = input('You: ')

    if msg != '':

        res = chatbot_response(msg)
        print("Siri: " + res + '\n')

    if msg.lower() == 'bye':
        kill_siri = True

You: hi
Raw prediction: [4.5908961e-05 9.7240460e-01 2.8850671e-03 1.7391183e-06 4.6518680e-06
 1.2440155e-05 7.3545285e-05 3.9204683e-06 1.0529271e-02 1.8933296e-03
 2.8940258e-05 5.1231000e-06 7.6200429e-04 1.1184353e-05 5.3797495e-03
 1.3083154e-03 5.4615972e-05 4.1192383e-05 5.7002973e-05 2.1740110e-05
 5.1885785e-04 2.6434584e-04 6.0537275e-07 1.1738971e-04 2.2193191e-04
 2.7251325e-03 8.4701947e-05 7.8177160e-05 3.7238604e-04 7.8651647e-05
 1.3556718e-05]
Max probability: 0.9724046
Siri: Good to see you again

You: how r u
Raw prediction: [2.0756612e-05 2.8810835e-01 1.5625476e-03 1.0745890e-05 3.8364960e-05
 4.8147226e-03 1.6842561e-05 2.4937715e-05 1.0398184e-03 5.4034692e-01
 6.1006314e-05 7.1328373e-05 8.8471093e-04 6.1913568e-05 1.8405155e-04
 1.4094804e-01 4.8263354e-04 2.3121282e-04 3.1818702e-06 2.4630828e-05
 1.5941755e-03 1.6045724e-03 6.2912345e-06 8.9196488e-04 7.3583105e-05
 2.1134472e-05 6.2107213e-04 6.2038400e-03 1.3810779e-05 9.6542633e-04
 9.0674302e-03]
Max pro

In [34]:
import ipywidgets as widgets
from IPython.display import display, HTML
from datetime import datetime

# Chat display area
chat_area = widgets.VBox(
    layout=widgets.Layout(
        width="500px",
        height="400px",
        overflow_y="auto",
        border="1px solid #ddd",
        padding="10px",
        background_color="#f9f9f9"
    )
)

# Input box
text_box = widgets.Text(
    placeholder="Type your message...",
    layout=widgets.Layout(width="400px")
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    layout=widgets.Layout(width="90px")
)

# Store messages
messages = []

def create_bubble(text, sender):
    color = "#DCF8C6" if sender == "You" else "#FFFFFF"
    align = "flex-end" if sender == "You" else "flex-start"

    bubble = widgets.HTML(
        value=f"""
        <div style="
            display:flex;
            justify-content:{align};
            margin:5px 0;">
            <div style="
                background:{color};
                padding:10px;
                border-radius:10px;
                max-width:70%;
                box-shadow:0px 1px 3px rgba(0,0,0,0.2);
                font-family:Arial;
                font-size:14px;">
                <b>{sender}:</b> {text}
            </div>
        </div>
        """
    )
    return bubble

def update_chat():
    chat_area.children = messages

def on_send(_):
    user_msg = text_box.value.strip()
    if not user_msg:
        return

    # User message
    messages.append(create_bubble(user_msg, "You"))

    # Bot response
    bot_msg = chatbot_response(user_msg)
    messages.append(create_bubble(bot_msg, "Siri"))

    # Update UI
    update_chat()

    text_box.value = ""

send_button.on_click(on_send)

# Layout
ui = widgets.VBox([
    chat_area,
    widgets.HBox([text_box, send_button])
])

display(ui)

Raw prediction: [4.5908961e-05 9.7240460e-01 2.8850671e-03 1.7391183e-06 4.6518680e-06
 1.2440155e-05 7.3545285e-05 3.9204683e-06 1.0529271e-02 1.8933296e-03
 2.8940258e-05 5.1231000e-06 7.6200429e-04 1.1184353e-05 5.3797495e-03
 1.3083154e-03 5.4615972e-05 4.1192383e-05 5.7002973e-05 2.1740110e-05
 5.1885785e-04 2.6434584e-04 6.0537275e-07 1.1738971e-04 2.2193191e-04
 2.7251325e-03 8.4701947e-05 7.8177160e-05 3.7238604e-04 7.8651647e-05
 1.3556718e-05]
Max probability: 0.9724046


In [ ]:
import ipywidgets
import notebook

print("ipywidgets:", ipywidgets.__version__)
print("notebook:", notebook.__version__)

ipywidgets: 7.7.1
notebook: 6.5.7


In [ ]:
pip install --upgrade ipywidgets notebook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.8/910.8 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    

In [37]:
import nbformat

file = "/content/sample_data/Chatbot.ipynb"  # replace with your filename

nb = nbformat.read(file, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, file)

print("Widget metadata removed.")

Widget metadata removed.


In [38]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'chatbot_model.h5', 'sample_data']
